# BARseq VISp ETL

Register the 1,448-cell QC cohort, soma coordinates, projection matrices and BARseq taxonomy under project `barseq_visp` and dataset `barseq_visp_qc`.
Inputs are the flat files produced by [BARseq_data_extraction_01.ipynb](BARseq_data_extraction_01.ipynb).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models, write_projection_matrix
from connects_common_connectivity.io.write_utils import walk_ancestors
from connects_common_connectivity.models import (
    AlgorithmRun,
    AnatomicalOrientation,
    CellToClusterMapping,
    Cluster,
    ClusterHierarchy,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    HierarchyCategory,
    Laterality,
    LocationType,
    MappingSet,
    Modality,
    ProjectionMeasurementMatrix,
    ProjectionMeasurementType,
    ReferenceSpace,
    SignedAxis,
    SpatialLocation,
    Unit,
)


In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').is_file()
)
INPUT_ROOT = REPO_ROOT / 'data/BARseq/extracted'
OUTPUT_ROOT = get_settings().output_root
PROJECT_ID = 'barseq_visp'
DATASET_ID = 'barseq_visp_qc'
PUBLICATION = 'https://www.biorxiv.org/content/10.64898/2026.09.01.748541v2'
HIERARCHY_ID = 'barseq_visp_inferred_taxonomy'
RUN_ID = 'barseq_visp_ingest_01'
ROOT_ID = 'cell'
REFERENCE_SPACE_ID = 'ccfv3_25um'
TASIC_HIERARCHY_ID = 'tasic_2018_visp_taxonomy'
TASIC_MAPPING_SET_ID = 'barseq_visp_xck12_tasic_mapping'
TASIC_ALIASES = {'L6b VISp Col8a1 Rprm': 'L6b Col8a1 Rprm'}
ABC_HIERARCHY_ID = 'CCN20230722'
ABC_MAPPING_SPECS = [
    ('xck12', 750, [
        ('supertype', 'abc_supertype', 'abc_supertype_conf'),
        ('cluster', 'abc_cluster', 'abc_cluster_conf'),
    ]),
    ('xck17', 834, [
        ('subclass', 'abc_subclass', 'subclass_conf'),
        ('cluster', 'abc_cluster', 'cluster_conf'),
    ]),
]
# Brain 824690: four unsampled targets plus LP, which has only limited sampling.
MASKED_TARGETS = ['VISa', 'SSp-bfd', 'LGd', 'LD', 'LP']
PROJECTION_SPECS = [
    ('barseq_proj_18_total', 'projections_targets_18_total.csv', Laterality.BILATERAL.value),
    ('barseq_proj_18_ipsi', 'projections_targets_18_ipsi.csv', Laterality.IPSILATERAL.value),
    ('barseq_proj_18_contra', 'projections_targets_18_contra.csv', Laterality.CONTRALATERAL.value),
    ('barseq_proj_429_total', 'projections_ccf_429.csv', Laterality.BILATERAL.value),
]
assert INPUT_ROOT.is_dir(), 'Run BARseq_data_extraction_01.ipynb first.'
assert (OUTPUT_ROOT / 'cluster' / '_delta_log').is_dir(), 'Run etl_ttype_01.ipynb first.'
print('Input:', INPUT_ROOT, '\nOutput:', OUTPUT_ROOT)


Input: /workspaces/ConnectsCommonConnectivity/data/BARseq/extracted 
Output: /workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1


## Load source tables

In [3]:
neurons = pd.read_csv(INPUT_ROOT / 'neurons.csv', dtype={'uid': str, 'brain': str})
soma = pd.read_csv(INPUT_ROOT / 'soma_ccf.csv', dtype={'uid': str}).set_index('uid')
coverage = pd.read_csv(INPUT_ROOT / 'brain_target_coverage.csv', dtype={'brain': str})
provenance = json.loads((INPUT_ROOT / 'provenance.json').read_text())
targets = pd.read_csv(INPUT_ROOT / 'projection_targets.csv')['target'].tolist()
cell_ids = neurons['uid'].tolist()
masked_ids = neurons.loc[neurons['brain'].eq('824690'), 'uid']

assert len(cell_ids) == len(set(cell_ids)) == 1448
assert neurons['nm1_qc_pass'].eq(True).all()
assert neurons['brain'].value_counts().to_dict() == {'777552': 1035, '824690': 413}
assert neurons['uid'].str.split('_').str[0].eq(neurons['brain']).all()
assert soma.index.is_unique and set(soma.index) == set(cell_ids)
soma = soma.loc[cell_ids]
assert np.isfinite(soma.to_numpy()).all()

projections = {}
for matrix_id, filename, _ in PROJECTION_SPECS:
    frame = pd.read_csv(INPUT_ROOT / filename, dtype={'uid': str}).set_index('uid')
    assert frame.index.is_unique and set(frame.index) == set(cell_ids)
    assert frame.columns.is_unique
    values = frame.to_numpy(dtype='float64')
    assert np.isfinite(values).all() and (values >= 0).all()
    assert np.equal(values, np.floor(values)).all()
    if matrix_id == 'barseq_proj_429_total':
        # The 18-target coverage flags do not define a mask for individual CCF leaf regions.
        assert frame.shape == (1448, 429)
    else:
        assert frame.columns.tolist() == targets and len(targets) == 18
        # NaN marks unavailable measurements; Arrow converts these to stored nulls.
        frame = frame.astype('float64')
        frame.loc[masked_ids, MASKED_TARGETS] = np.nan
    projections[matrix_id] = frame.loc[cell_ids]

for name, frame in {'neurons': neurons, 'soma': soma, **projections}.items():
    print(name, frame.shape)
    print(frame.iloc[:3, :6])


neurons (1448, 7)
                  uid   brain  nm1_qc_pass  tx_hq  verified_isbc  cell_row
0  777552_1_33_150279  777552         True  False           True      9137
1  777552_1_33_150484  777552         True  False           True      9331
2  777552_1_33_150545  777552         True   True           True      9392
soma (1448, 3)
                        ap_vox     dv_vox      ml_vox
uid                                                  
777552_1_33_150279  376.917359  74.456951  104.928023
777552_1_33_150484  377.484045  69.065299  101.032184
777552_1_33_150545  377.603152  67.932079   99.405005
barseq_proj_18_total (1448, 18)
                    VISpor  VISpl  VISli  VISl  VISal  VISrl
uid                                                         
777552_1_33_150279    14.0    3.0    0.0   2.0    0.0    0.0
777552_1_33_150484     6.0    0.0    1.0  12.0    6.0    0.0
777552_1_33_150545    10.0    9.0    0.0   5.0    0.0    0.0
barseq_proj_18_ipsi (1448, 18)
                    VISpor  V

## Dataset and DataItems

In [4]:
dataset = DataSet(
    id=DATASET_ID, project_id=PROJECT_ID,
    name='BARseq VISp: 1,448 QC neurons from brains 777552 and 824690',
    publication=PUBLICATION, modality=Modality.BARCODED.value,
)
items = [DataItem(id=uid, name=uid, project_id=PROJECT_ID) for uid in cell_ids]
associations = [
    DataItemDataSetAssociation(dataitem_id=uid, dataset_id=DATASET_ID, project_id=PROJECT_ID)
    for uid in cell_ids
]
for rows in ([dataset], items, associations):
    print(write_models(rows, output_root=OUTPUT_ROOT))


WrittenResult(class_name='DataSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/dataset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=1)
WrittenResult(class_name='DataItem', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/dataitem'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=1448)
WrittenResult(class_name='DataItemDataSetAssociation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/dataitem_dataset_association'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataset_id" = source."dataset_id" AND target."dataitem_id" = source."dataitem_id" AND target."project_id" IN (\'barseq_visp\')',)

In [5]:
project_scope = pl.col('project_id') == PROJECT_ID
stored_dataset = pl.read_delta(OUTPUT_ROOT / 'dataset').filter(
    project_scope & (pl.col('id') == DATASET_ID)
)
stored_items = pl.read_delta(OUTPUT_ROOT / 'dataitem').filter(project_scope)
stored_associations = pl.read_delta(OUTPUT_ROOT / 'dataitem_dataset_association').filter(
    project_scope & (pl.col('dataset_id') == DATASET_ID)
)
assert stored_dataset.height == 1 and stored_dataset.item(0, 'publication') == PUBLICATION
assert stored_items.height == stored_associations.height == 1448
assert set(stored_items['id']) == set(stored_associations['dataitem_id']) == set(cell_ids)
for stored in (stored_dataset, stored_items, stored_associations):
    print(stored.shape, stored.head(3))


(1, 5) shape: (1, 5)
┌────────────────┬───────────────────────┬────────────────────────────────┬──────────┬─────────────┐
│ id             ┆ name                  ┆ publication                    ┆ modality ┆ project_id  │
│ ---            ┆ ---                   ┆ ---                            ┆ ---      ┆ ---         │
│ str            ┆ str                   ┆ str                            ┆ str      ┆ str         │
╞════════════════╪═══════════════════════╪════════════════════════════════╪══════════╪═════════════╡
│ barseq_visp_qc ┆ BARseq VISp: 1,448 QC ┆ https://www.biorxiv.org/conten ┆ BARCODED ┆ barseq_visp │
│                ┆ neurons …             ┆ …                              ┆          ┆             │
└────────────────┴───────────────────────┴────────────────────────────────┴──────────┴─────────────┘
(1448, 4) shape: (3, 4)
┌────────────────────┬────────────────────┬───────────────────┬─────────────┐
│ id                 ┆ name               ┆ neuroglancer_link ┆ proje

## Soma coordinates

Store unchanged CCFv3 25-um voxel coordinates with x=AP, y=DV and z=ML.

In [6]:
reference_space = ReferenceSpace(
    id=REFERENCE_SPACE_ID, project_id=PROJECT_ID, name='CCFv3 25-um voxel coordinates',
    unit=Unit.VOXELS.value, voxel_size=[25.0, 25.0, 25.0],
    voxel_size_unit=Unit.MICRONS_LENGTH.value,
    anatomical_orientation=AnatomicalOrientation(
        anterior_to_posterior=SignedAxis.PLUS_X.value,
        dorsal_to_ventral=SignedAxis.PLUS_Y.value,
        subject_left_to_right=SignedAxis.PLUS_Z.value,
    ),
)
locations = [
    SpatialLocation(
        project_id=PROJECT_ID, dataitem_id=uid, reference_space=REFERENCE_SPACE_ID,
        location_type=LocationType.SOMA.value,
        x=row.ap_vox, y=row.dv_vox, z=row.ml_vox,
    )
    for uid, row in soma.iterrows()
]
print(write_models([reference_space], output_root=OUTPUT_ROOT))
print(write_models(locations, output_root=OUTPUT_ROOT))


WrittenResult(class_name='ReferenceSpace', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/referencespace'), mode='merge_scoped', predicates=('(target."project_id" IS NOT DISTINCT FROM source."project_id") AND target."id" = source."id"',), rows_written=1)
WrittenResult(class_name='SpatialLocation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/spatiallocation'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataitem_id" = source."dataitem_id" AND target."reference_space" = source."reference_space" AND target."location_type" = source."location_type" AND target."project_id" IN (\'barseq_visp\')',), rows_written=1448)


In [7]:
stored_space = pl.read_delta(OUTPUT_ROOT / 'referencespace').filter(
    project_scope & (pl.col('id') == REFERENCE_SPACE_ID)
)
stored_locations = pl.read_delta(OUTPUT_ROOT / 'spatiallocation').filter(
    project_scope & (pl.col('reference_space') == REFERENCE_SPACE_ID)
    & (pl.col('location_type') == LocationType.SOMA.value)
)
assert stored_space.height == 1 and stored_space.item(0, 'unit') == Unit.VOXELS.value
assert stored_locations.height == 1448 and set(stored_locations['dataitem_id']) == set(cell_ids)
np.testing.assert_array_equal(
    stored_locations.sort('dataitem_id').select('x', 'y', 'z').to_numpy(),
    soma.sort_index()[['ap_vox', 'dv_vox', 'ml_vox']].to_numpy(),
)
for stored in (stored_space, stored_locations):
    print(stored.shape, stored.head(3))


(1, 9) shape: (1, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ project_i ┆ id        ┆ name      ┆ descripti ┆ … ┆ voxel_siz ┆ voxel_siz ┆ default_2 ┆ anatomic │
│ d         ┆ ---       ┆ ---       ┆ on        ┆   ┆ e         ┆ e_unit    ┆ d_view    ┆ al_orien │
│ ---       ┆ str       ┆ str       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ tation   │
│ str       ┆           ┆           ┆ str       ┆   ┆ list[f64] ┆ str       ┆ struct[2] ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ struct[7 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ ]        │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ barseq_vi ┆ ccfv3_25u ┆ CCFv3     ┆ null      ┆ … ┆ [25.0,    ┆ MICRONS_L ┆ null      ┆ {"PLUS_X │
│ sp        ┆ m         ┆ 25-um     ┆           ┆   ┆ 25.0,     ┆ ENGT

## BARseq taxonomy, memberships and provenance

Import the extracted class/subclass/cluster tree: 10 classes, 46 subclasses and 83 observed leaves, plus a root.
Register all 954 labeled cells; the ingestion run retains source provenance, cell QC flags and per-brain target coverage.

In [8]:
classes = pd.read_csv(INPUT_ROOT / 'taxonomy_classes.csv')['class'].tolist()
subclasses = pd.read_csv(INPUT_ROOT / 'taxonomy_subclasses.csv')
leaves = pd.read_csv(INPUT_ROOT / 'taxonomy_clusters.csv')
colors = pd.read_csv(INPUT_ROOT / 'subclass_colours.csv', keep_default_na=False)
colors = dict(zip(colors['subclass'], colors['hex_colour']))
subclass_parent = dict(zip(subclasses['subclass'], subclasses['class']))
leaf_parent = dict(zip(leaves['cluster'], leaves['subclass']))
assert len(classes) == len(set(classes)) == 10
assert len(subclass_parent) == len(subclasses) == 46
assert len(leaf_parent) == len(leaves) == 83
assert set(subclass_parent.values()) <= set(classes)
assert set(leaf_parent.values()) <= set(subclass_parent)

# Category levels run leaf-to-root; Cluster.level measures depth from the root.
categories = [
    HierarchyCategory(id=name, hierarchy_id=HIERARCHY_ID, level=level)
    for level, name in enumerate(['cluster', 'subclass', 'class', 'major_class'])
]
# Prefix node IDs by level: labels such as TH occur at both subclass and cluster levels.
clusters = [Cluster(
    id=ROOT_ID, hierarchy_id=HIERARCHY_ID, children=[f'class:{label}' for label in classes],
    level=0, hierarchy_category='major_class',
)]
clusters.extend(
    Cluster(
        id=f'class:{label}', hierarchy_id=HIERARCHY_ID, parent=ROOT_ID, level=1,
        children=[f'subclass:{c}' for c, parent in subclass_parent.items() if parent == label],
        hierarchy_category='class',
    )
    for label in classes
)
clusters.extend(
    Cluster(
        id=f'subclass:{label}', hierarchy_id=HIERARCHY_ID, parent=f'class:{parent}', level=2,
        children=[f'cluster:{child}' for child, owner in leaf_parent.items() if owner == label],
        hierarchy_category='subclass', hex_color=colors[label] or None,
    )
    for label, parent in subclass_parent.items()
)
clusters.extend(
    Cluster(
        id=f'cluster:{label}', hierarchy_id=HIERARCHY_ID,
        parent=f'subclass:{parent}', level=3, children=[],
        hierarchy_category='cluster', hex_color=colors[parent] or None,
    )
    for label, parent in leaf_parent.items()
)
assert len({cluster.id for cluster in clusters}) == 140
hierarchy = ClusterHierarchy(
    id=HIERARCHY_ID, run=RUN_ID, root=ROOT_ID, clusters=[cluster.id for cluster in clusters],
)

assignments = pd.read_csv(
    INPUT_ROOT / 'neuron_taxonomy.csv', dtype={'uid': str}, keep_default_na=False,
)
assert assignments['uid'].is_unique and set(assignments['uid']) == set(cell_ids)
# Keep every source label; NA is unassigned, and tx_hq is metadata rather than a filter.
labeled = assignments.loc[assignments['cluster'].ne('NA')]
assert len(labeled) == 954 and set(labeled['cluster']) <= set(leaf_parent)
assert labeled['cluster'].map(leaf_parent).eq(labeled['subclass']).all()
parent_of = {cluster.id: cluster.parent for cluster in clusters}
# Include ancestors so consumers can query memberships at any taxonomy level.
memberships = [
    ClusterMembership(
        project_id=PROJECT_ID, hierarchy_id=HIERARCHY_ID, item=row.uid, cluster=cluster_id,
    )
    for row in labeled.itertuples(index=False)
    for cluster_id, _ in walk_ancestors(f'cluster:{row.cluster}', parent_of)
]
assert len(memberships) == 954 * 4

# Preserve per-cell QC and sampling coverage separately from the measurement values.
ingest_metadata = {
    'source_provenance': provenance,
    'membership_policy': 'all_labeled',
    'cohort_qc_flags': neurons.to_dict(orient='records'),
    'brain_target_coverage': coverage.to_dict(orient='records'),
}
ingest_run = AlgorithmRun(
    id=RUN_ID, algorithm_name='BARseq extracted-cohort and inferred-taxonomy import',
    input_dataset=DATASET_ID, json_object=json.dumps(ingest_metadata, allow_nan=False),
)


In [9]:
for rows in ([ingest_run], categories, clusters, [hierarchy], memberships):
    print(write_models(rows, output_root=OUTPUT_ROOT))


WrittenResult(class_name='AlgorithmRun', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/algorithmrun'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)
WrittenResult(class_name='HierarchyCategory', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/hierarchycategory'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'barseq_visp_inferred_taxonomy\')',), rows_written=4)
WrittenResult(class_name='Cluster', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/cluster'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'barseq_visp_inferred_taxonomy\')',), rows_written=140)
WrittenResult(class_name='ClusterHierarchy', path=PosixPath('/workspaces/ConnectsCommonConnectivity/sc

In [10]:
taxonomy_scope = pl.col('hierarchy_id') == HIERARCHY_ID
stored_clusters = pl.read_delta(OUTPUT_ROOT / 'cluster').filter(taxonomy_scope)
stored_categories = pl.read_delta(OUTPUT_ROOT / 'hierarchycategory').filter(taxonomy_scope)
stored_hierarchy = pl.read_delta(OUTPUT_ROOT / 'clusterhierarchy').filter(
    pl.col('id') == HIERARCHY_ID
)
stored_run = pl.read_delta(OUTPUT_ROOT / 'algorithmrun').filter(pl.col('id') == RUN_ID)
assert stored_clusters.height == 140 and stored_categories.height == 4
assert stored_hierarchy.height == stored_run.height == 1
assert set(stored_clusters['id']) == set(hierarchy.clusters)
assert set(stored_clusters['parent'].drop_nulls()) <= set(stored_clusters['id'])
assert dict(zip(stored_clusters['id'], stored_clusters['parent'])) == parent_of
stored_memberships = pl.read_delta(OUTPUT_ROOT / 'clustermembership').filter(
    project_scope & taxonomy_scope
)
assert stored_memberships.height == 3816 and stored_memberships['item'].n_unique() == 954
assert set(stored_memberships.select('item', 'cluster').iter_rows()) == {
    (row.item, row.cluster) for row in memberships
}
assert json.loads(stored_run.item(0, 'json_object')) == ingest_metadata
for stored in (
    stored_clusters, stored_categories, stored_hierarchy, stored_run, stored_memberships,
):
    print(stored.shape, stored.head(3))


(140, 9) shape: (3, 9)
┌────────────┬────────────┬────────────┬───────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ parent     ┆ children   ┆ level ┆ … ┆ hex_color ┆ hierarchy ┆ distance_ ┆ hierarchy │
│ ---        ┆ ---        ┆ ---        ┆ ---   ┆   ┆ ---       ┆ _category ┆ to_parent ┆ _id       │
│ str        ┆ str        ┆ list[str]  ┆ i64   ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆            ┆       ┆   ┆           ┆ str       ┆ f64       ┆ str       │
╞════════════╪════════════╪════════════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ class:Hypo ┆ cell       ┆ ["subclass ┆ 1     ┆ … ┆ null      ┆ class     ┆ null      ┆ barseq_vi │
│ thalamus   ┆            ┆ :HY"]      ┆       ┆   ┆           ┆           ┆           ┆ sp_inferr │
│            ┆            ┆            ┆       ┆   ┆           ┆           ┆           ┆ ed_taxono │
│            ┆            ┆            ┆       ┆   ┆           ┆    

## Tasic mappings

Map Harmony xck12 predictions to the existing Tasic 2018 VISp hierarchy, reusing its colors without modifying the taxonomy.
Rename only the source label `L6b VISp Col8a1 Rprm` to `L6b Col8a1 Rprm` (17 cells); all other labels stay unchanged.

In [11]:
tasic_clusters = pl.read_delta(OUTPUT_ROOT / 'cluster').filter(
    pl.col('hierarchy_id') == TASIC_HIERARCHY_ID
)
tasic_hierarchy = pl.read_delta(OUTPUT_ROOT / 'clusterhierarchy').filter(
    pl.col('id') == TASIC_HIERARCHY_ID
)
assert tasic_hierarchy.height == 1, 'Run etl_ttype_01.ipynb first.'
tasic_parent = dict(zip(tasic_clusters['id'], tasic_clusters['parent']))
tasic_leaves = set(tasic_clusters.filter(pl.col('hierarchy_category') == 'cluster')['id'])
harmony = pd.read_csv(INPUT_ROOT / 'harmony_xck12.csv', dtype={'uid': str})
assert len(harmony) == harmony['uid'].nunique() == 750
assert set(harmony['uid']) <= set(cell_ids)
# This is an exact alias, not a general removal of VISp from source labels.
harmony['target_cluster'] = harmony['tasic_cluster'].replace(TASIC_ALIASES)
assert harmony['target_cluster'].ne(harmony['tasic_cluster']).sum() == 17
assert set(harmony['target_cluster']) <= tasic_leaves
assert harmony['tasic_cluster_conf'].between(0, 1).all()

tasic_mapping_set = MappingSet(
    id=TASIC_MAPPING_SET_ID, project_id=PROJECT_ID,
    name='BARseq VISp Harmony xck12 Tasic assignments',
    method_name='Harmony reference mapping (xck12)',
    source_dataset=DATASET_ID, target_hierarchy=TASIC_HIERARCHY_ID,
    json_object=json.dumps({
        'source': 'harmony_xck12.csv', 'label_aliases': TASIC_ALIASES,
        'score': 'Source tasic_cluster_conf, retained on leaf mappings only.',
    }),
)
# Preserve all predictions and their source confidence; do not score ancestor mappings.
tasic_mappings = [
    CellToClusterMapping(
        id=f'{row.uid}-{cluster_id}-{TASIC_MAPPING_SET_ID}',
        project_id=PROJECT_ID, mapping_set=TASIC_MAPPING_SET_ID,
        source_cell=row.uid, target_cluster=cluster_id,
        score=row.tasic_cluster_conf if is_leaf else None,
    )
    for row in harmony.itertuples(index=False)
    for cluster_id, is_leaf in walk_ancestors(row.target_cluster, tasic_parent)
]
assert len(tasic_mappings) == 3000


In [12]:
print(write_models([tasic_mapping_set], output_root=OUTPUT_ROOT))
print(write_models(tasic_mappings, output_root=OUTPUT_ROOT))


WrittenResult(class_name='MappingSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/mappingset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=1)
WrittenResult(class_name='CellToClusterMapping', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/celltoclustermapping'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."mapping_set" = source."mapping_set" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=3000)


In [13]:
stored_mapping_set = pl.read_delta(OUTPUT_ROOT / 'mappingset').filter(
    project_scope & (pl.col('id') == TASIC_MAPPING_SET_ID)
)
stored_tasic_mappings = pl.read_delta(OUTPUT_ROOT / 'celltoclustermapping').filter(
    project_scope & (pl.col('mapping_set') == TASIC_MAPPING_SET_ID)
)
assert stored_mapping_set.height == 1
assert stored_mapping_set.item(0, 'target_hierarchy') == TASIC_HIERARCHY_ID
assert stored_tasic_mappings.height == 3000
assert set(stored_tasic_mappings['source_cell']) == set(harmony['uid'])
assert set(stored_tasic_mappings.select('source_cell', 'target_cluster', 'score').iter_rows()) == {
    (row.source_cell, row.target_cluster, row.score) for row in tasic_mappings
}
# Adding mappings must not change the shared Tasic nodes or their colors.
assert pl.read_delta(OUTPUT_ROOT / 'cluster').filter(
    pl.col('hierarchy_id') == TASIC_HIERARCHY_ID
).sort('id').equals(tasic_clusters.sort('id'))
for stored in (stored_mapping_set, stored_tasic_mappings):
    print(stored.shape, stored.head(3))


(1, 13) shape: (1, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ name      ┆ descripti ┆ method_na ┆ … ┆ source_hi ┆ target_hi ┆ json_obje ┆ project_ │
│ ---       ┆ ---       ┆ on        ┆ me        ┆   ┆ erarchy   ┆ erarchy   ┆ ct        ┆ id       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ barseq_vi ┆ BARseq    ┆ null      ┆ Harmony   ┆ … ┆ null      ┆ tasic_201 ┆ {"source" ┆ barseq_v │
│ sp_xck12_ ┆ VISp      ┆           ┆ reference ┆   ┆           ┆ 8_visp_ta ┆ : "harmon ┆ isp      │
│ tasic_map ┆ Harmony   ┆           ┆ mapping   ┆   ┆           ┆ xonomy    ┆ y_xck12.c ┆          │
│ pin…      ┆ xck12     ┆           ┆ (xck…     ┆   ┆           ┆   

## ABC mappings

Keep xck12 and xck17 predictions in separate mapping sets targeting `CCN20230722`; run [etl_abc_01.ipynb](../cell_types/etl_abc_01.ipynb) first.
Store each supplied level and confidence directly, without adding ancestors or reconciling disagreements between predictions.

In [14]:
abc_hierarchy = pl.read_delta(OUTPUT_ROOT / 'clusterhierarchy').filter(
    pl.col('id') == ABC_HIERARCHY_ID
)
assert abc_hierarchy.height == 1, 'Run etl_abc_01.ipynb first.'
abc_clusters = pl.read_delta(OUTPUT_ROOT / 'cluster').filter(
    pl.col('hierarchy_id') == ABC_HIERARCHY_ID
)
abc_mapping_sets = []
abc_mappings = []
for run_name, expected_cells, fields in ABC_MAPPING_SPECS:
    filename = f'harmony_{run_name}.csv'
    predictions = pd.read_csv(INPUT_ROOT / filename, dtype={'uid': str})
    assert len(predictions) == predictions['uid'].nunique() == expected_cells
    assert set(predictions['uid']) <= set(cell_ids)
    mapping_set_id = f'barseq_visp_{run_name}_abc_mapping'
    abc_mapping_sets.append(MappingSet(
        id=mapping_set_id, project_id=PROJECT_ID,
        name=f'BARseq VISp Harmony {run_name} ABC assignments',
        method_name=f'Harmony reference mapping ({run_name})',
        source_dataset=DATASET_ID, target_hierarchy=ABC_HIERARCHY_ID,
        json_object=json.dumps({
            'source': filename, 'prediction_fields': fields,
            'score': 'Source confidence for each independently predicted level.',
            'ancestor_mappings': False,
        }),
    ))
    for category, label_column, confidence_column in fields:
        targets = category + ':' + predictions[label_column]
        registered = set(abc_clusters.filter(pl.col('hierarchy_category') == category)['id'])
        assert targets.notna().all() and set(targets) <= registered
        assert predictions[confidence_column].between(0, 1).all()
        # Direct predictions may disagree with the cluster's ancestors; preserve both.
        abc_mappings.extend(
            CellToClusterMapping(
                id=f'{uid}-{target}-{mapping_set_id}',
                project_id=PROJECT_ID, mapping_set=mapping_set_id,
                source_cell=uid, target_cluster=target, score=confidence,
            )
            for uid, target, confidence in zip(
                predictions['uid'], targets, predictions[confidence_column],
            )
        )
assert len(abc_mappings) == len({row.id for row in abc_mappings}) == 3168


In [15]:
print(write_models(abc_mapping_sets, output_root=OUTPUT_ROOT))
print(write_models(abc_mappings, output_root=OUTPUT_ROOT))


WrittenResult(class_name='MappingSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/mappingset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=2)
WrittenResult(class_name='CellToClusterMapping', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/celltoclustermapping'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."mapping_set" = source."mapping_set" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=3168)


In [16]:
abc_set_ids = [row.id for row in abc_mapping_sets]
stored_abc_sets = pl.read_delta(OUTPUT_ROOT / 'mappingset').filter(
    project_scope & pl.col('id').is_in(abc_set_ids)
)
stored_abc_mappings = pl.read_delta(OUTPUT_ROOT / 'celltoclustermapping').filter(
    project_scope & pl.col('mapping_set').is_in(abc_set_ids)
)
assert stored_abc_sets.height == 2
assert set(stored_abc_sets['target_hierarchy']) == {ABC_HIERARCHY_ID}
assert set(stored_abc_sets['source_dataset']) == {DATASET_ID}
assert stored_abc_mappings.height == 3168
assert set(stored_abc_mappings.select(
    'mapping_set', 'source_cell', 'target_cluster', 'score',
).iter_rows()) == {
    (row.mapping_set, row.source_cell, row.target_cluster, row.score) for row in abc_mappings
}
for run_name, expected_cells, fields in ABC_MAPPING_SPECS:
    stored = stored_abc_mappings.filter(
        pl.col('mapping_set') == f'barseq_visp_{run_name}_abc_mapping'
    )
    assert stored.height == expected_cells * len(fields)
    assert stored['source_cell'].n_unique() == expected_cells
assert pl.read_delta(OUTPUT_ROOT / 'cluster').filter(
    pl.col('hierarchy_id') == ABC_HIERARCHY_ID
).sort('id').equals(abc_clusters.sort('id'))
for stored in (stored_abc_sets, stored_abc_mappings):
    print(stored.shape, stored.head(3))


(2, 13) shape: (2, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ name      ┆ descripti ┆ method_na ┆ … ┆ source_hi ┆ target_hi ┆ json_obje ┆ project_ │
│ ---       ┆ ---       ┆ on        ┆ me        ┆   ┆ erarchy   ┆ erarchy   ┆ ct        ┆ id       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ barseq_vi ┆ BARseq    ┆ null      ┆ Harmony   ┆ … ┆ null      ┆ CCN202307 ┆ {"source" ┆ barseq_v │
│ sp_xck12_ ┆ VISp      ┆           ┆ reference ┆   ┆           ┆ 22        ┆ : "harmon ┆ isp      │
│ abc_mappi ┆ Harmony   ┆           ┆ mapping   ┆   ┆           ┆           ┆ y_xck12.c ┆          │
│ ng        ┆ xck12 ABC ┆           ┆ (xck…     ┆   ┆           ┆   

## Projection matrices

Write rolony counts in the WNM layout, with nulls for brain 824690's unsampled VISa, SSp-bfd, LGd and LD entries and limited-sampling LP entries in the three 18-target matrices.
The 18-target total includes the midline band excluded from ipsi/contra; the 429-region total retains `NA` and VISp bins.
The 429-region matrix remains unchanged; sampling coverage is retained in the ingestion run.

In [17]:
projection_paths = {}
for matrix_id, _, laterality in PROJECTION_SPECS:
    frame = projections[matrix_id]
    regions = frame.columns.tolist()
    wide = frame.rename_axis('id').reset_index()
    wide.insert(1, 'project_id', PROJECT_ID)
    wide.insert(2, 'dataset_id', DATASET_ID)
    schema = pa.schema([
        pa.field('id', pa.string(), nullable=False),
        pa.field('project_id', pa.string(), nullable=False),
        pa.field('dataset_id', pa.string(), nullable=False),
        *[pa.field(region, pa.float64(), nullable=True) for region in regions],
    ])
    payload_path = OUTPUT_ROOT / 'projectionmeasurementmatrix' / matrix_id
    sampling_note = (
        '429-region source counts, unmasked.' if matrix_id == 'barseq_proj_429_total'
        else '18-target counts; 824690 VISa, SSp-bfd, LGd, LD and LP are null (sampling mask).'
    )
    matrix = ProjectionMeasurementMatrix(
        id=matrix_id, project_id=PROJECT_ID,
        description=(
            f'BARseq VISp rolony counts. {sampling_note} Coverage: AlgorithmRun {RUN_ID}. '
            '824690: VISa, SSp-bfd, LGd, LD unsampled; LP limited. Contra imaging medial-only.'
        ),
        measurement_type=ProjectionMeasurementType.DETECTIONS.value,
        modality=Modality.BARCODED.value, unit=Unit.COUNT.value, laterality=laterality,
        region_index=regions, data_item_index=cell_ids,
        values=payload_path.relative_to(OUTPUT_ROOT).as_posix(),
    )
    write_deltalake(
        payload_path, pa.Table.from_pandas(wide, schema=schema, preserve_index=False),
        mode='overwrite', predicate=f"project_id = '{PROJECT_ID}'", partition_by=['project_id'],
    )
    # Only detected signal contributes to region_coverage; payload nulls remain unchanged.
    print(write_projection_matrix(matrix, frame.gt(0).to_numpy(), output_root=OUTPUT_ROOT))
    projection_paths[matrix_id] = payload_path


WrittenResult(class_name='ProjectionMeasurementMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/projectionmeasurementmatrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=1)
WrittenResult(class_name='ProjectionMeasurementMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/projectionmeasurementmatrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'barseq_visp\')',), rows_written=1)
WrittenResult(class_name='ProjectionMeasurementMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/projectionmeasurementmatrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'barseq

In [18]:
stored_matrices = pl.read_delta(OUTPUT_ROOT / 'projectionmeasurementmatrix').filter(
    project_scope & pl.col('id').is_in(list(projection_paths))
)
assert stored_matrices.height == 4
print(stored_matrices.shape, stored_matrices.head(3))
for matrix_id, _, laterality in PROJECTION_SPECS:
    frame = projections[matrix_id]
    stored = pl.read_delta(projection_paths[matrix_id]).filter(project_scope)
    assert stored.shape == (1448, frame.shape[1] + 3)
    assert stored['id'].n_unique() == 1448 and set(stored['id']) == set(cell_ids)
    assert set(stored['dataset_id']) == {DATASET_ID}
    regions = frame.columns.tolist()
    assert stored.select(regions).null_count().row(0) == tuple(frame.isna().sum())
    assert int(frame.isna().sum().sum()) == (0 if frame.shape[1] == 429 else 413 * 5)
    np.testing.assert_array_equal(
        stored.sort('id').select(frame.columns.tolist()).to_numpy(),
        frame.sort_index().to_numpy(),
    )
    metadata = stored_matrices.filter(pl.col('id') == matrix_id).row(0, named=True)
    assert metadata['laterality'] == laterality
    assert metadata['region_index'] == frame.columns.tolist()
    assert metadata['data_item_index'] == cell_ids
    assert metadata['region_coverage'] == frame.columns[frame.gt(0).any()].tolist()
    print(matrix_id, stored.shape, stored.select('id', *frame.columns[:3]).head(3))


(4, 11) shape: (3, 11)
┌────────────┬────────────┬────────────┬──────────┬───┬────────────┬───────────┬───────┬───────────┐
│ id         ┆ descriptio ┆ measuremen ┆ modality ┆ … ┆ data_item_ ┆ values    ┆ unit  ┆ project_i │
│ ---        ┆ n          ┆ t_type     ┆ ---      ┆   ┆ index      ┆ ---       ┆ ---   ┆ d         │
│ str        ┆ ---        ┆ ---        ┆ str      ┆   ┆ ---        ┆ str       ┆ str   ┆ ---       │
│            ┆ str        ┆ str        ┆          ┆   ┆ list[str]  ┆           ┆       ┆ str       │
╞════════════╪════════════╪════════════╪══════════╪═══╪════════════╪═══════════╪═══════╪═══════════╡
│ barseq_pro ┆ BARseq     ┆ DETECTIONS ┆ BARCODED ┆ … ┆ ["777552_1 ┆ file:///w ┆ COUNT ┆ barseq_vi │
│ j_429_tota ┆ VISp       ┆            ┆          ┆   ┆ _33_150279 ┆ orkspaces ┆       ┆ sp        │
│ l          ┆ rolony     ┆            ┆          ┆   ┆ ",         ┆ /Connects ┆       ┆           │
│            ┆ counts.    ┆            ┆          ┆   ┆ "777552…   ┆

## Outputs

Paths are relative to `OUTPUT_ROOT`; row counts refer to this dataset or hierarchy.

| Path | Rows |
| --- | ---: |
| `dataset/` | 1 |
| `dataitem/` | 1,448 |
| `dataitem_dataset_association/` | 1,448 |
| `referencespace/` | 1 |
| `spatiallocation/` | 1,448 |
| `algorithmrun/` | 1 |
| `hierarchycategory/` | 4 |
| `cluster/` | 140 |
| `clusterhierarchy/` | 1 |
| `clustermembership/` | 3,816 (954 cells, each with a leaf and three ancestors) |
| `mappingset/` | 3 (Tasic, ABC xck12, ABC xck17) |
| `celltoclustermapping/` | 6,168 (3,000 Tasic + 1,500 ABC xck12 + 1,668 ABC xck17) |
| `projectionmeasurementmatrix/` | 4 |
| `projectionmeasurementmatrix/barseq_proj_18_total/` | 1,448 x 18 measurements |
| `projectionmeasurementmatrix/barseq_proj_18_ipsi/` | 1,448 x 18 measurements |
| `projectionmeasurementmatrix/barseq_proj_18_contra/` | 1,448 x 18 measurements |
| `projectionmeasurementmatrix/barseq_proj_429_total/` | 1,448 x 429 measurements |